<a href="https://colab.research.google.com/github/ced-sys/SubTerra/blob/main/Site_Comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install rasterio

In [ ]:
# Site Embeddings and Imagery Analysis Pipeline
# Comprehensive analysis of site changes between 2023 and 2024

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Scientific computing and ML libraries
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
import scipy.stats as stats

# Image processing
try:
    import rasterio
    import rasterio.features
    from rasterio.enums import Resampling
    RASTERIO_AVAILABLE = True
except ImportError:
    print("Warning: rasterio not available. Install with: pip install rasterio")
    RASTERIO_AVAILABLE = False

# Mount Google Drive (uncomment if using Colab)
from google.colab import drive
drive.mount('/content/drive')

# Configuration
base_path = "/content/drive/MyDrive/Company folders"  # Update this path
years = [2023, 2024]
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

class SiteAnalyzer:
    def __init__(self, base_path, years=[2023, 2024]):
        self.base_path = Path(base_path)
        self.years = years
        self.embeddings = None
        self.image_features = None
        self.changes_df = None
        self.combined_data = None

    def load_embeddings(self):
        """Load and combine embeddings from all sites and years"""
        print("Loading embeddings...")
        all_dfs = []

        for year in self.years:
            folder = self.base_path / str(year) / "embeddings"
            if not folder.exists():
                print(f"Warning: Folder {folder} not found")
                continue

            csv_files = list(folder.glob("*.csv"))
            print(f"Found {len(csv_files)} embedding files for {year}")

            for file_path in csv_files:
                try:
                    site_id = file_path.stem  # filename without extension
                    df = pd.read_csv(file_path)

                    # Handle different embedding formats
                    if df.shape[0] == 1:
                        # Single row embedding
                        df['site_id'] = site_id
                        df['year'] = year
                        all_dfs.append(df)
                    else:
                        # Multiple rows - take mean (could be patch-level embeddings)
                        numeric_cols = df.select_dtypes(include=[np.number]).columns
                        emb_mean = df[numeric_cols].mean().to_frame().T
                        emb_mean['site_id'] = site_id
                        emb_mean['year'] = year
                        all_dfs.append(emb_mean)

                except Exception as e:
                    print(f"Error loading {file_path}: {e}")

        if all_dfs:
            self.embeddings = pd.concat(all_dfs, ignore_index=True)
            print(f"Loaded embeddings: {self.embeddings.shape}")
            return self.embeddings
        else:
            print("No embeddings loaded!")
            return None

    def extract_advanced_image_features(self, tif_path):
        """Extract comprehensive features from TIF images"""
        if not RASTERIO_AVAILABLE:
            return None

        try:
            with rasterio.open(tif_path) as src:
                data = src.read()  # Shape: (bands, height, width)

                features = {}

                for band_idx, band in enumerate(data):
                    band_name = f"band_{band_idx}"

                    # Basic statistics
                    features[f'{band_name}_mean'] = np.mean(band)
                    features[f'{band_name}_std'] = np.std(band)
                    features[f'{band_name}_median'] = np.median(band)
                    features[f'{band_name}_min'] = np.min(band)
                    features[f'{band_name}_max'] = np.max(band)
                    features[f'{band_name}_range'] = np.ptp(band)

                    # Distribution statistics
                    features[f'{band_name}_skew'] = stats.skew(band.flatten())
                    features[f'{band_name}_kurtosis'] = stats.kurtosis(band.flatten())

                    # Percentiles
                    features[f'{band_name}_p10'] = np.percentile(band, 10)
                    features[f'{band_name}_p25'] = np.percentile(band, 25)
                    features[f'{band_name}_p75'] = np.percentile(band, 75)
                    features[f'{band_name}_p90'] = np.percentile(band, 90)

                # Cross-band features if multiple bands
                if len(data) > 1:
                    # Band ratios (useful for vegetation indices, etc.)
                    for i in range(len(data)):
                        for j in range(i+1, len(data)):
                            ratio = np.divide(data[i], data[j], out=np.ones_like(data[i]), where=data[j]!=0)
                            features[f'ratio_band{i}_band{j}'] = np.mean(ratio)

                    # Cross-band correlation
                    flat_bands = [band.flatten() for band in data]
                    for i in range(len(flat_bands)):
                        for j in range(i+1, len(flat_bands)):
                            corr = np.corrcoef(flat_bands[i], flat_bands[j])[0,1]
                            features[f'corr_band{i}_band{j}'] = corr if not np.isnan(corr) else 0

                # Spatial features (texture measures)
                for band_idx, band in enumerate(data):
                    # Gradient magnitude (edge detection)
                    gy, gx = np.gradient(band.astype(float))
                    grad_mag = np.sqrt(gx**2 + gy**2)
                    features[f'band_{band_idx}_gradient_mean'] = np.mean(grad_mag)
                    features[f'band_{band_idx}_gradient_std'] = np.std(grad_mag)

                return features

        except Exception as e:
            print(f"Error extracting features from {tif_path}: {e}")
            return None

    def load_image_features(self):
        """Load and extract features from all TIF images"""
        print("Extracting image features...")
        all_features = []

        for year in self.years:
            folder = self.base_path / str(year) / "images"
            if not folder.exists():
                print(f"Warning: Folder {folder} not found")
                continue

            tif_files = list(folder.glob("*.tif")) + list(folder.glob("*.TIF"))
            print(f"Found {len(tif_files)} image files for {year}")

            for file_path in tif_files:
                site_id = file_path.stem
                features = self.extract_advanced_image_features(file_path)

                if features:
                    features['site_id'] = site_id
                    features['year'] = year
                    all_features.append(features)

        if all_features:
            self.image_features = pd.DataFrame(all_features)
            print(f"Extracted image features: {self.image_features.shape}")
            return self.image_features
        else:
            print("No image features extracted!")
            return None

    def compute_temporal_changes(self):
        """Compute various change metrics between years"""
        print("Computing temporal changes...")

        if self.embeddings is None:
            print("No embeddings loaded!")
            return None

        changes = []
        embedding_cols = [col for col in self.embeddings.columns if col not in ['site_id', 'year']]

        for site in self.embeddings['site_id'].unique():
            site_data = self.embeddings[self.embeddings['site_id'] == site]

            # Get embeddings for each year
            embeddings_by_year = {}
            for year in self.years:
                year_data = site_data[site_data['year'] == year]
                if not year_data.empty:
                    embeddings_by_year[year] = year_data[embedding_cols].values[0]

            # Compute changes between consecutive years
            for i in range(len(self.years) - 1):
                year1, year2 = self.years[i], self.years[i + 1]

                if year1 in embeddings_by_year and year2 in embeddings_by_year:
                    emb1 = embeddings_by_year[year1].reshape(1, -1)
                    emb2 = embeddings_by_year[year2].reshape(1, -1)

                    # Similarity metrics
                    cos_sim = cosine_similarity(emb1, emb2)[0][0]
                    eucl_dist = euclidean_distances(emb1, emb2)[0][0]

                    # Manhattan distance
                    manhattan_dist = np.sum(np.abs(emb1 - emb2))

                    # Relative change magnitude
                    relative_change = eucl_dist / (np.linalg.norm(emb1) + 1e-10)

                    changes.append({
                        'site_id': site,
                        'year_from': year1,
                        'year_to': year2,
                        'cosine_similarity': cos_sim,
                        'euclidean_distance': eucl_dist,
                        'manhattan_distance': manhattan_dist,
                        'relative_change': relative_change,
                        'cosine_distance': 1 - cos_sim
                    })

        if changes:
            self.changes_df = pd.DataFrame(changes)
            print(f"Computed {len(changes)} site changes")
            return self.changes_df
        else:
            print("No changes computed!")
            return None

    def combine_datasets(self):
        """Combine embeddings and image features for joint analysis"""
        print("Combining datasets...")

        datasets_to_combine = []

        if self.embeddings is not None:
            emb_pivot = self.embeddings.set_index(['site_id', 'year'])
            emb_cols = [col for col in self.embeddings.columns if col not in ['site_id', 'year']]
            emb_pivot.columns = [f'emb_{col}' for col in emb_cols]
            datasets_to_combine.append(emb_pivot)

        if self.image_features is not None:
            img_pivot = self.image_features.set_index(['site_id', 'year'])
            img_cols = [col for col in self.image_features.columns if col not in ['site_id', 'year']]
            img_pivot.columns = [f'img_{col}' for col in img_cols]
            datasets_to_combine.append(img_pivot)

        if datasets_to_combine:
            self.combined_data = pd.concat(datasets_to_combine, axis=1).reset_index()
            print(f"Combined dataset: {self.combined_data.shape}")
            return self.combined_data
        else:
            print("No datasets to combine!")
            return None

    def run_clustering_analysis(self, n_clusters=3):
        """Perform clustering analysis on combined data"""
        print(f"Running clustering analysis with {n_clusters} clusters...")

        if self.combined_data is None:
            print("No combined data available!")
            return None

        # Prepare data for clustering
        feature_cols = [col for col in self.combined_data.columns if col not in ['site_id', 'year']]
        X = self.combined_data[feature_cols].fillna(0)

        # Standardize features
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        # K-means clustering
        kmeans = KMeans(n_clusters=n_clusters, random_state=42)
        clusters = kmeans.fit_predict(X_scaled)

        # Add cluster labels to data
        result_df = self.combined_data.copy()
        result_df['cluster'] = clusters

        return result_df, X_scaled, scaler

    def run_pca_analysis(self, n_components=None):
        """Perform PCA analysis"""
        print("Running PCA analysis...")

        if self.combined_data is None:
            print("No combined data available!")
            return None

        feature_cols = [col for col in self.combined_data.columns if col not in ['site_id', 'year']]
        X = self.combined_data[feature_cols].fillna(0)

        if n_components is None:
            n_components = min(10, X.shape[1])

        # Standardize and run PCA
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        pca = PCA(n_components=n_components)
        X_pca = pca.fit_transform(X_scaled)

        # Create results dataframe
        pca_cols = [f'PC{i+1}' for i in range(n_components)]
        pca_df = pd.DataFrame(X_pca, columns=pca_cols)
        pca_df['site_id'] = self.combined_data['site_id'].values
        pca_df['year'] = self.combined_data['year'].values

        return pca_df, pca, scaler

    def create_comprehensive_plots(self):
        """Create comprehensive visualization suite"""
        print("Creating visualizations...")

        # Set up the plotting style
        plt.rcParams['figure.figsize'] = (15, 10)

        # 1. Temporal Changes Overview
        if self.changes_df is not None:
            fig, axes = plt.subplots(2, 2, figsize=(15, 12))
            fig.suptitle('Temporal Changes Analysis (2023 → 2024)', fontsize=16, fontweight='bold')

            # Euclidean distance distribution
            axes[0,0].hist(self.changes_df['euclidean_distance'], bins=15, alpha=0.7, color='skyblue', edgecolor='black')
            axes[0,0].set_title('Distribution of Euclidean Distances')
            axes[0,0].set_xlabel('Euclidean Distance')
            axes[0,0].set_ylabel('Frequency')

            # Cosine similarity distribution
            axes[0,1].hist(self.changes_df['cosine_similarity'], bins=15, alpha=0.7, color='lightgreen', edgecolor='black')
            axes[0,1].set_title('Distribution of Cosine Similarities')
            axes[0,1].set_xlabel('Cosine Similarity')
            axes[0,1].set_ylabel('Frequency')

            # Per-site changes
            axes[1,0].bar(range(len(self.changes_df)), self.changes_df['euclidean_distance'], alpha=0.7, color='coral')
            axes[1,0].set_title('Per-Site Euclidean Distance Changes')
            axes[1,0].set_xlabel('Site Index')
            axes[1,0].set_ylabel('Euclidean Distance')
            axes[1,0].tick_params(axis='x', rotation=45)

            # Scatter plot of different distance metrics
            axes[1,1].scatter(self.changes_df['euclidean_distance'], self.changes_df['cosine_distance'],
                            alpha=0.7, s=100, color='purple')
            axes[1,1].set_xlabel('Euclidean Distance')
            axes[1,1].set_ylabel('Cosine Distance')
            axes[1,1].set_title('Euclidean vs Cosine Distance')

            plt.tight_layout()
            plt.show()

            # Summary statistics
            print("\n=== TEMPORAL CHANGES SUMMARY ===")
            print(self.changes_df.describe())

        # 2. PCA Analysis
        pca_results = self.run_pca_analysis()
        if pca_results is not None:
            pca_df, pca, scaler = pca_results

            fig, axes = plt.subplots(2, 2, figsize=(15, 12))
            fig.suptitle('Principal Component Analysis', fontsize=16, fontweight='bold')

            # Explained variance
            axes[0,0].bar(range(1, len(pca.explained_variance_ratio_) + 1),
                         pca.explained_variance_ratio_, alpha=0.7, color='gold')
            axes[0,0].set_title('Explained Variance by Principal Component')
            axes[0,0].set_xlabel('Principal Component')
            axes[0,0].set_ylabel('Explained Variance Ratio')

            # Cumulative explained variance
            cumsum_var = np.cumsum(pca.explained_variance_ratio_)
            axes[0,1].plot(range(1, len(cumsum_var) + 1), cumsum_var, 'o-', color='red', linewidth=2)
            axes[0,1].set_title('Cumulative Explained Variance')
            axes[0,1].set_xlabel('Number of Components')
            axes[0,1].set_ylabel('Cumulative Explained Variance')
            axes[0,1].grid(True, alpha=0.3)

            # PC1 vs PC2 scatter plot
            colors = ['red' if year == 2023 else 'blue' for year in pca_df['year']]
            axes[1,0].scatter(pca_df['PC1'], pca_df['PC2'], c=colors, alpha=0.7, s=100)
            axes[1,0].set_xlabel('PC1')
            axes[1,0].set_ylabel('PC2')
            axes[1,0].set_title('PC1 vs PC2 (Red=2023, Blue=2024)')
            axes[1,0].grid(True, alpha=0.3)

            # Site trajectories in PC space
            for site in pca_df['site_id'].unique():
                site_data = pca_df[pca_df['site_id'] == site]
                if len(site_data) > 1:
                    axes[1,1].plot(site_data['PC1'], site_data['PC2'], 'o-', alpha=0.7, linewidth=2)
            axes[1,1].set_xlabel('PC1')
            axes[1,1].set_ylabel('PC2')
            axes[1,1].set_title('Site Trajectories in PC Space')
            axes[1,1].grid(True, alpha=0.3)

            plt.tight_layout()
            plt.show()

        # 3. Clustering Analysis
        clustering_results = self.run_clustering_analysis()
        if clustering_results is not None:
            cluster_df, X_scaled, scaler = clustering_results

            fig, axes = plt.subplots(2, 2, figsize=(15, 12))
            fig.suptitle('Clustering Analysis', fontsize=16, fontweight='bold')

            # Cluster distribution
            cluster_counts = cluster_df['cluster'].value_counts().sort_index()
            axes[0,0].bar(cluster_counts.index, cluster_counts.values, alpha=0.7, color='lightcoral')
            axes[0,0].set_title('Cluster Size Distribution')
            axes[0,0].set_xlabel('Cluster')
            axes[0,0].set_ylabel('Number of Sites')

            # Clusters in PC space (if PCA was run)
            if pca_results is not None:
                pca_df_with_clusters = pca_df.copy()
                pca_df_with_clusters['cluster'] = cluster_df['cluster']

                scatter = axes[0,1].scatter(pca_df_with_clusters['PC1'], pca_df_with_clusters['PC2'],
                                          c=pca_df_with_clusters['cluster'], cmap='viridis', alpha=0.7, s=100)
                axes[0,1].set_xlabel('PC1')
                axes[0,1].set_ylabel('PC2')
                axes[0,1].set_title('Clusters in PC Space')
                plt.colorbar(scatter, ax=axes[0,1])

            # Cluster composition by year
            cluster_year = pd.crosstab(cluster_df['cluster'], cluster_df['year'])
            cluster_year.plot(kind='bar', ax=axes[1,0], alpha=0.7)
            axes[1,0].set_title('Cluster Composition by Year')
            axes[1,0].set_xlabel('Cluster')
            axes[1,0].set_ylabel('Count')
            axes[1,0].legend(title='Year')

            # Feature importance for clustering (using cluster centers)
            if hasattr(self, 'combined_data'):
                feature_cols = [col for col in self.combined_data.columns if col not in ['site_id', 'year']]
                # This is a simplified feature importance based on cluster center distances
                axes[1,1].text(0.1, 0.5, 'Cluster Centers Analysis\n(Feature importance visualization\nwould require additional\nfeature selection methods)',
                              transform=axes[1,1].transAxes, fontsize=12)
                axes[1,1].set_title('Feature Importance (Placeholder)')

            plt.tight_layout()
            plt.show()

    def generate_summary_report(self):
        """Generate a comprehensive summary report"""
        print("\n" + "="*60)
        print("COMPREHENSIVE SITE ANALYSIS REPORT")
        print("="*60)

        if self.embeddings is not None:
            print(f"\n📊 EMBEDDINGS DATA:")
            print(f"   • Total sites: {self.embeddings['site_id'].nunique()}")
            print(f"   • Years covered: {sorted(self.embeddings['year'].unique())}")
            print(f"   • Embedding dimensions: {self.embeddings.shape[1] - 2}")  # -2 for site_id, year
            print(f"   • Total observations: {len(self.embeddings)}")

        if self.image_features is not None:
            print(f"\n🖼️  IMAGE FEATURES:")
            print(f"   • Total images processed: {len(self.image_features)}")
            print(f"   • Feature dimensions: {self.image_features.shape[1] - 2}")  # -2 for site_id, year

        if self.changes_df is not None:
            print(f"\n📈 TEMPORAL CHANGES:")
            print(f"   • Site comparisons: {len(self.changes_df)}")
            print(f"   • Mean euclidean distance: {self.changes_df['euclidean_distance'].mean():.4f}")
            print(f"   • Mean cosine similarity: {self.changes_df['cosine_similarity'].mean():.4f}")
            print(f"   • Most changed site: {self.changes_df.loc[self.changes_df['euclidean_distance'].idxmax(), 'site_id']}")
            print(f"   • Least changed site: {self.changes_df.loc[self.changes_df['euclidean_distance'].idxmin(), 'site_id']}")

        if self.combined_data is not None:
            print(f"\n🔗 COMBINED DATASET:")
            print(f"   • Total features: {self.combined_data.shape[1] - 2}")
            print(f"   • Data completeness: {(1 - self.combined_data.isnull().sum().sum() / self.combined_data.size) * 100:.1f}%")

        print(f"\n✅ Analysis completed successfully!")
        print("="*60)

    def run_full_analysis(self):
        """Run the complete analysis pipeline"""
        print("Starting comprehensive site analysis...")

        # Load data
        self.load_embeddings()
        if RASTERIO_AVAILABLE:
            self.load_image_features()
        else:
            print("Skipping image analysis (rasterio not available)")

        # Compute changes
        self.compute_temporal_changes()

        # Combine datasets
        self.combine_datasets()

        # Create visualizations
        self.create_comprehensive_plots()

        # Generate report
        self.generate_summary_report()

        return {
            'embeddings': self.embeddings,
            'image_features': self.image_features,
            'changes': self.changes_df,
            'combined_data': self.combined_data
        }

# Usage Example
if __name__ == "__main__":
    # Initialize analyzer
    analyzer = SiteAnalyzer(base_path=base_path, years=[2023, 2024])

    # Run complete analysis
    results = analyzer.run_full_analysis()

    # Access individual results
    # embeddings_df = results['embeddings']
    # changes_df = results['changes']
    # combined_df = results['combined_data']

    print("\nAnalysis complete! Results stored in 'results' dictionary.")

In [ ]:
# Site Embeddings and Imagery Analysis Pipeline
# Comprehensive analysis of site changes between 2023 and 2024

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Scientific computing and ML libraries
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
import scipy.stats as stats

# Image processing
try:
    import rasterio
    import rasterio.features
    from rasterio.enums import Resampling
    RASTERIO_AVAILABLE = True
except ImportError:
    print("Warning: rasterio not available. Install with: pip install rasterio")
    RASTERIO_AVAILABLE = False

# Mount Google Drive (uncomment if using Colab)
# from google.colab import drive
# drive.mount('/content/drive')

# Configuration
base_path = "/content/drive/MyDrive/Company folders"  # Update this path
years = [2023, 2024]
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

class SiteAnalyzer:
    def __init__(self, base_path, years=[2023, 2024]):
        self.base_path = Path(base_path)
        self.years = years
        self.embeddings = None
        self.image_features = None
        self.changes_df = None
        self.combined_data = None

    def load_embeddings(self):
        """Load and combine embeddings from all sites and years"""
        print("Loading embeddings...")
        all_dfs = []

        for year in self.years:
            # Updated folder structure: "2023 embeddings (CSV)", "2024 embeddings (CSV)", etc.
            folder = self.base_path / f"{year} embeddings (CSV)"
            if not folder.exists():
                print(f"Warning: Folder {folder} not found")
                continue

            csv_files = list(folder.glob("*.csv"))
            print(f"Found {len(csv_files)} embedding files for {year}")

            for file_path in csv_files:
                try:
                    site_id = file_path.stem  # filename without extension
                    df = pd.read_csv(file_path)

                    # Handle different embedding formats
                    if df.shape[0] == 1:
                        # Single row embedding
                        df['site_id'] = site_id
                        df['year'] = year
                        all_dfs.append(df)
                    else:
                        # Multiple rows - take mean (could be patch-level embeddings)
                        numeric_cols = df.select_dtypes(include=[np.number]).columns
                        emb_mean = df[numeric_cols].mean().to_frame().T
                        emb_mean['site_id'] = site_id
                        emb_mean['year'] = year
                        all_dfs.append(emb_mean)

                except Exception as e:
                    print(f"Error loading {file_path}: {e}")

        if all_dfs:
            self.embeddings = pd.concat(all_dfs, ignore_index=True)
            print(f"Loaded embeddings: {self.embeddings.shape}")
            return self.embeddings
        else:
            print("No embeddings loaded!")
            return None

    def extract_advanced_image_features(self, tif_path):
        """Extract comprehensive features from TIF images"""
        if not RASTERIO_AVAILABLE:
            return None

        try:
            with rasterio.open(tif_path) as src:
                data = src.read()  # Shape: (bands, height, width)

                features = {}

                for band_idx, band in enumerate(data):
                    band_name = f"band_{band_idx}"

                    # Basic statistics
                    features[f'{band_name}_mean'] = np.mean(band)
                    features[f'{band_name}_std'] = np.std(band)
                    features[f'{band_name}_median'] = np.median(band)
                    features[f'{band_name}_min'] = np.min(band)
                    features[f'{band_name}_max'] = np.max(band)
                    features[f'{band_name}_range'] = np.ptp(band)

                    # Distribution statistics
                    features[f'{band_name}_skew'] = stats.skew(band.flatten())
                    features[f'{band_name}_kurtosis'] = stats.kurtosis(band.flatten())

                    # Percentiles
                    features[f'{band_name}_p10'] = np.percentile(band, 10)
                    features[f'{band_name}_p25'] = np.percentile(band, 25)
                    features[f'{band_name}_p75'] = np.percentile(band, 75)
                    features[f'{band_name}_p90'] = np.percentile(band, 90)

                # Cross-band features if multiple bands
                if len(data) > 1:
                    # Band ratios (useful for vegetation indices, etc.)
                    for i in range(len(data)):
                        for j in range(i+1, len(data)):
                            ratio = np.divide(data[i], data[j], out=np.ones_like(data[i]), where=data[j]!=0)
                            features[f'ratio_band{i}_band{j}'] = np.mean(ratio)

                    # Cross-band correlation
                    flat_bands = [band.flatten() for band in data]
                    for i in range(len(flat_bands)):
                        for j in range(i+1, len(flat_bands)):
                            corr = np.corrcoef(flat_bands[i], flat_bands[j])[0,1]
                            features[f'corr_band{i}_band{j}'] = corr if not np.isnan(corr) else 0

                # Spatial features (texture measures)
                for band_idx, band in enumerate(data):
                    # Gradient magnitude (edge detection)
                    gy, gx = np.gradient(band.astype(float))
                    grad_mag = np.sqrt(gx**2 + gy**2)
                    features[f'band_{band_idx}_gradient_mean'] = np.mean(grad_mag)
                    features[f'band_{band_idx}_gradient_std'] = np.std(grad_mag)

                return features

        except Exception as e:
            print(f"Error extracting features from {tif_path}: {e}")
            return None

    def load_image_features(self):
        """Load and extract features from all TIF images"""
        print("Extracting image features...")
        all_features = []

        for year in self.years:
            # Updated folder structure: "2023 embeddings (TIF)", "2024 embeddings (TIF)", etc.
            folder = self.base_path / f"{year} embeddings (TIF)"
            if not folder.exists():
                print(f"Warning: Folder {folder} not found")
                continue

            tif_files = list(folder.glob("*.tif")) + list(folder.glob("*.TIF"))
            print(f"Found {len(tif_files)} image files for {year}")

            for file_path in tif_files:
                site_id = file_path.stem
                features = self.extract_advanced_image_features(file_path)

                if features:
                    features['site_id'] = site_id
                    features['year'] = year
                    all_features.append(features)

        if all_features:
            self.image_features = pd.DataFrame(all_features)
            print(f"Extracted image features: {self.image_features.shape}")
            return self.image_features
        else:
            print("No image features extracted!")
            return None

    def compute_temporal_changes(self):
        """Compute various change metrics between years"""
        print("Computing temporal changes...")

        if self.embeddings is None:
            print("No embeddings loaded!")
            return None

        changes = []
        embedding_cols = [col for col in self.embeddings.columns if col not in ['site_id', 'year']]

        for site in self.embeddings['site_id'].unique():
            site_data = self.embeddings[self.embeddings['site_id'] == site]

            # Get embeddings for each year
            embeddings_by_year = {}
            for year in self.years:
                year_data = site_data[site_data['year'] == year]
                if not year_data.empty:
                    embeddings_by_year[year] = year_data[embedding_cols].values[0]

            # Compute changes between consecutive years
            for i in range(len(self.years) - 1):
                year1, year2 = self.years[i], self.years[i + 1]

                if year1 in embeddings_by_year and year2 in embeddings_by_year:
                    emb1 = embeddings_by_year[year1].reshape(1, -1)
                    emb2 = embeddings_by_year[year2].reshape(1, -1)

                    # Similarity metrics
                    cos_sim = cosine_similarity(emb1, emb2)[0][0]
                    eucl_dist = euclidean_distances(emb1, emb2)[0][0]

                    # Manhattan distance
                    manhattan_dist = np.sum(np.abs(emb1 - emb2))

                    # Relative change magnitude
                    relative_change = eucl_dist / (np.linalg.norm(emb1) + 1e-10)

                    changes.append({
                        'site_id': site,
                        'year_from': year1,
                        'year_to': year2,
                        'cosine_similarity': cos_sim,
                        'euclidean_distance': eucl_dist,
                        'manhattan_distance': manhattan_dist,
                        'relative_change': relative_change,
                        'cosine_distance': 1 - cos_sim
                    })

        if changes:
            self.changes_df = pd.DataFrame(changes)
            print(f"Computed {len(changes)} site changes")
            return self.changes_df
        else:
            print("No changes computed!")
            return None

    def combine_datasets(self):
        """Combine embeddings and image features for joint analysis"""
        print("Combining datasets...")

        datasets_to_combine = []

        if self.embeddings is not None:
            emb_pivot = self.embeddings.set_index(['site_id', 'year'])
            emb_cols = [col for col in self.embeddings.columns if col not in ['site_id', 'year']]
            emb_pivot.columns = [f'emb_{col}' for col in emb_cols]
            datasets_to_combine.append(emb_pivot)

        if self.image_features is not None:
            img_pivot = self.image_features.set_index(['site_id', 'year'])
            img_cols = [col for col in self.image_features.columns if col not in ['site_id', 'year']]
            img_pivot.columns = [f'img_{col}' for col in img_cols]
            datasets_to_combine.append(img_pivot)

        if datasets_to_combine:
            self.combined_data = pd.concat(datasets_to_combine, axis=1).reset_index()
            print(f"Combined dataset: {self.combined_data.shape}")
            return self.combined_data
        else:
            print("No datasets to combine!")
            return None

    def run_clustering_analysis(self, n_clusters=3):
        """Perform clustering analysis on combined data"""
        print(f"Running clustering analysis with {n_clusters} clusters...")

        if self.combined_data is None:
            print("No combined data available!")
            return None

        # Prepare data for clustering
        feature_cols = [col for col in self.combined_data.columns if col not in ['site_id', 'year']]
        X = self.combined_data[feature_cols].fillna(0)

        # Standardize features
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        # K-means clustering
        kmeans = KMeans(n_clusters=n_clusters, random_state=42)
        clusters = kmeans.fit_predict(X_scaled)

        # Add cluster labels to data
        result_df = self.combined_data.copy()
        result_df['cluster'] = clusters

        return result_df, X_scaled, scaler

    def run_pca_analysis(self, n_components=None):
        """Perform PCA analysis"""
        print("Running PCA analysis...")

        if self.combined_data is None:
            print("No combined data available!")
            return None

        feature_cols = [col for col in self.combined_data.columns if col not in ['site_id', 'year']]
        X = self.combined_data[feature_cols].fillna(0)

        if n_components is None:
            n_components = min(10, X.shape[1])

        # Standardize and run PCA
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        pca = PCA(n_components=n_components)
        X_pca = pca.fit_transform(X_scaled)

        # Create results dataframe
        pca_cols = [f'PC{i+1}' for i in range(n_components)]
        pca_df = pd.DataFrame(X_pca, columns=pca_cols)
        pca_df['site_id'] = self.combined_data['site_id'].values
        pca_df['year'] = self.combined_data['year'].values

        return pca_df, pca, scaler

    def create_comprehensive_plots(self):
        """Create comprehensive visualization suite"""
        print("Creating visualizations...")

        # Set up the plotting style
        plt.rcParams['figure.figsize'] = (15, 10)

        # 1. Temporal Changes Overview
        if self.changes_df is not None:
            fig, axes = plt.subplots(2, 2, figsize=(15, 12))
            fig.suptitle('Temporal Changes Analysis (2023 → 2024)', fontsize=16, fontweight='bold')

            # Euclidean distance distribution
            axes[0,0].hist(self.changes_df['euclidean_distance'], bins=15, alpha=0.7, color='skyblue', edgecolor='black')
            axes[0,0].set_title('Distribution of Euclidean Distances')
            axes[0,0].set_xlabel('Euclidean Distance')
            axes[0,0].set_ylabel('Frequency')

            # Cosine similarity distribution
            axes[0,1].hist(self.changes_df['cosine_similarity'], bins=15, alpha=0.7, color='lightgreen', edgecolor='black')
            axes[0,1].set_title('Distribution of Cosine Similarities')
            axes[0,1].set_xlabel('Cosine Similarity')
            axes[0,1].set_ylabel('Frequency')

            # Per-site changes
            axes[1,0].bar(range(len(self.changes_df)), self.changes_df['euclidean_distance'], alpha=0.7, color='coral')
            axes[1,0].set_title('Per-Site Euclidean Distance Changes')
            axes[1,0].set_xlabel('Site Index')
            axes[1,0].set_ylabel('Euclidean Distance')
            axes[1,0].tick_params(axis='x', rotation=45)

            # Scatter plot of different distance metrics
            axes[1,1].scatter(self.changes_df['euclidean_distance'], self.changes_df['cosine_distance'],
                            alpha=0.7, s=100, color='purple')
            axes[1,1].set_xlabel('Euclidean Distance')
            axes[1,1].set_ylabel('Cosine Distance')
            axes[1,1].set_title('Euclidean vs Cosine Distance')

            plt.tight_layout()
            plt.show()

            # Summary statistics
            print("\n=== TEMPORAL CHANGES SUMMARY ===")
            print(self.changes_df.describe())

        # 2. PCA Analysis
        pca_results = self.run_pca_analysis()
        if pca_results is not None:
            pca_df, pca, scaler = pca_results

            fig, axes = plt.subplots(2, 2, figsize=(15, 12))
            fig.suptitle('Principal Component Analysis', fontsize=16, fontweight='bold')

            # Explained variance
            axes[0,0].bar(range(1, len(pca.explained_variance_ratio_) + 1),
                         pca.explained_variance_ratio_, alpha=0.7, color='gold')
            axes[0,0].set_title('Explained Variance by Principal Component')
            axes[0,0].set_xlabel('Principal Component')
            axes[0,0].set_ylabel('Explained Variance Ratio')

            # Cumulative explained variance
            cumsum_var = np.cumsum(pca.explained_variance_ratio_)
            axes[0,1].plot(range(1, len(cumsum_var) + 1), cumsum_var, 'o-', color='red', linewidth=2)
            axes[0,1].set_title('Cumulative Explained Variance')
            axes[0,1].set_xlabel('Number of Components')
            axes[0,1].set_ylabel('Cumulative Explained Variance')
            axes[0,1].grid(True, alpha=0.3)

            # PC1 vs PC2 scatter plot
            colors = ['red' if year == 2023 else 'blue' for year in pca_df['year']]
            axes[1,0].scatter(pca_df['PC1'], pca_df['PC2'], c=colors, alpha=0.7, s=100)
            axes[1,0].set_xlabel('PC1')
            axes[1,0].set_ylabel('PC2')
            axes[1,0].set_title('PC1 vs PC2 (Red=2023, Blue=2024)')
            axes[1,0].grid(True, alpha=0.3)

            # Site trajectories in PC space
            for site in pca_df['site_id'].unique():
                site_data = pca_df[pca_df['site_id'] == site]
                if len(site_data) > 1:
                    axes[1,1].plot(site_data['PC1'], site_data['PC2'], 'o-', alpha=0.7, linewidth=2)
            axes[1,1].set_xlabel('PC1')
            axes[1,1].set_ylabel('PC2')
            axes[1,1].set_title('Site Trajectories in PC Space')
            axes[1,1].grid(True, alpha=0.3)

            plt.tight_layout()
            plt.show()

        # 3. Clustering Analysis
        clustering_results = self.run_clustering_analysis()
        if clustering_results is not None:
            cluster_df, X_scaled, scaler = clustering_results

            fig, axes = plt.subplots(2, 2, figsize=(15, 12))
            fig.suptitle('Clustering Analysis', fontsize=16, fontweight='bold')

            # Cluster distribution
            cluster_counts = cluster_df['cluster'].value_counts().sort_index()
            axes[0,0].bar(cluster_counts.index, cluster_counts.values, alpha=0.7, color='lightcoral')
            axes[0,0].set_title('Cluster Size Distribution')
            axes[0,0].set_xlabel('Cluster')
            axes[0,0].set_ylabel('Number of Sites')

            # Clusters in PC space (if PCA was run)
            if pca_results is not None:
                pca_df_with_clusters = pca_df.copy()
                pca_df_with_clusters['cluster'] = cluster_df['cluster']

                scatter = axes[0,1].scatter(pca_df_with_clusters['PC1'], pca_df_with_clusters['PC2'],
                                          c=pca_df_with_clusters['cluster'], cmap='viridis', alpha=0.7, s=100)
                axes[0,1].set_xlabel('PC1')
                axes[0,1].set_ylabel('PC2')
                axes[0,1].set_title('Clusters in PC Space')
                plt.colorbar(scatter, ax=axes[0,1])

            # Cluster composition by year
            cluster_year = pd.crosstab(cluster_df['cluster'], cluster_df['year'])
            cluster_year.plot(kind='bar', ax=axes[1,0], alpha=0.7)
            axes[1,0].set_title('Cluster Composition by Year')
            axes[1,0].set_xlabel('Cluster')
            axes[1,0].set_ylabel('Count')
            axes[1,0].legend(title='Year')

            # Feature importance for clustering (using cluster centers)
            if hasattr(self, 'combined_data'):
                feature_cols = [col for col in self.combined_data.columns if col not in ['site_id', 'year']]
                # This is a simplified feature importance based on cluster center distances
                axes[1,1].text(0.1, 0.5, 'Cluster Centers Analysis\n(Feature importance visualization\nwould require additional\nfeature selection methods)',
                              transform=axes[1,1].transAxes, fontsize=12)
                axes[1,1].set_title('Feature Importance (Placeholder)')

            plt.tight_layout()
            plt.show()

    def generate_summary_report(self):
        """Generate a comprehensive summary report"""
        print("\n" + "="*60)
        print("COMPREHENSIVE SITE ANALYSIS REPORT")
        print("="*60)

        if self.embeddings is not None:
            print(f"\n📊 EMBEDDINGS DATA:")
            print(f"   • Total sites: {self.embeddings['site_id'].nunique()}")
            print(f"   • Years covered: {sorted(self.embeddings['year'].unique())}")
            print(f"   • Embedding dimensions: {self.embeddings.shape[1] - 2}")  # -2 for site_id, year
            print(f"   • Total observations: {len(self.embeddings)}")

        if self.image_features is not None:
            print(f"\n🖼️  IMAGE FEATURES:")
            print(f"   • Total images processed: {len(self.image_features)}")
            print(f"   • Feature dimensions: {self.image_features.shape[1] - 2}")  # -2 for site_id, year

        if self.changes_df is not None:
            print(f"\n📈 TEMPORAL CHANGES:")
            print(f"   • Site comparisons: {len(self.changes_df)}")
            print(f"   • Mean euclidean distance: {self.changes_df['euclidean_distance'].mean():.4f}")
            print(f"   • Mean cosine similarity: {self.changes_df['cosine_similarity'].mean():.4f}")
            print(f"   • Most changed site: {self.changes_df.loc[self.changes_df['euclidean_distance'].idxmax(), 'site_id']}")
            print(f"   • Least changed site: {self.changes_df.loc[self.changes_df['euclidean_distance'].idxmin(), 'site_id']}")

        if self.combined_data is not None:
            print(f"\n🔗 COMBINED DATASET:")
            print(f"   • Total features: {self.combined_data.shape[1] - 2}")
            print(f"   • Data completeness: {(1 - self.combined_data.isnull().sum().sum() / self.combined_data.size) * 100:.1f}%")

        print(f"\n✅ Analysis completed successfully!")
        print("="*60)

    def run_full_analysis(self):
        """Run the complete analysis pipeline"""
        print("Starting comprehensive site analysis...")

        # Load data
        self.load_embeddings()
        if RASTERIO_AVAILABLE:
            self.load_image_features()
        else:
            print("Skipping image analysis (rasterio not available)")

        # Compute changes
        self.compute_temporal_changes()

        # Combine datasets
        self.combine_datasets()

        # Create visualizations
        self.create_comprehensive_plots()

        # Generate report
        self.generate_summary_report()

        return {
            'embeddings': self.embeddings,
            'image_features': self.image_features,
            'changes': self.changes_df,
            'combined_data': self.combined_data
        }

# Usage Example
if __name__ == "__main__":
    # Initialize analyzer
    analyzer = SiteAnalyzer(base_path=base_path, years=[2023, 2024])

    # Run complete analysis
    results = analyzer.run_full_analysis()

    # Access individual results
    # embeddings_df = results['embeddings']
    # changes_df = results['changes']
    # combined_df = results['combined_data']

    print("\nAnalysis complete! Results stored in 'results' dictionary.")